<a href="https://colab.research.google.com/github/atulshine/BCC_APP/blob/Python_Docs/Working_On_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [119]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructField,StringType,StringType,IntegerType

In [110]:
abs_path = '/content/sample_data/organizations_data.csv'

In [111]:
df_csv=spark.read.format("csv").option("header","true").option("interSchema","true").load("/content/sample_data/organizations_data.csv")

In [112]:
df_csv.createOrReplaceTempView("temp_emp")

In [113]:
df_result=spark.sql("""SELECT * FROM temp_emp """)
df_result.show()

+-----+---------------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+-------------------+
|Index|Organization Id|                Name|             Website|             Country|         Description|Founded|            Industry|Number of employees|
+-----+---------------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+-------------------+
|    1|E84A904909dF528|          Liu-Hoover|http://www.day-ha...|      Western Sahara|Ergonomic zero ad...|   1980|   Online Publishing|               6852|
|    2|AAC4f9aBF86EAeF|       Orr-Armstrong|https://www.chapm...|             Algeria|Ergonomic radical...|   1970|     Import / Export|               7994|
|    3|ad2eb3C8C24DB87|           Gill-Lamb|     http://lin.com/|       Cote d'Ivoire|Programmable inte...|   2005|   Apparel / Fashion|               5105|
|    4|D76BB12E5eE165B|         Bauer-Weiss|https://gilles

In [125]:
highest_emp=spark.sql("""
SELECT * FROM
(SELECT temp_emp.*,dense_rank()over(partition by Country order by `Number of employees` desc)as rnk FROM temp_emp)
WHERE rnk=1
""")
highest_emp.show()

+-----+---------------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+-------------------+---+
|Index|Organization Id|                Name|             Website|             Country|         Description|Founded|            Industry|Number of employees|rnk|
+-----+---------------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+-------------------+---+
|   13|4EB9d3E5cF79b91|Bernard, Payne an...|http://williamson...|         Afghanistan|Polarized dynamic...|   1990|      Transportation|                730|  1|
|  578|3D41753AADAc2FB|       Mueller Group|https://www.nicho...|             Albania|Function-based va...|   1972|Graphic Design / ...|                956|  1|
|    2|AAC4f9aBF86EAeF|       Orr-Armstrong|https://www.chapm...|             Algeria|Ergonomic radical...|   1970|     Import / Export|               7994|  1|
|  467|2FaB6dAd4AfE024| Marquez-Bl

In [114]:
Source_count = df_csv.count()
Target_count = df_csv.count()

display(Source_count)
display(Target_count)

1000

1000

In [116]:
df_nulls = df_csv.filter(F.col("Organization Id").isNull() | F.isnan(F.col("Organization Id")))



In [117]:
total_rows = df_csv.count()
distinct_rows = df_csv.select("Organization Id").distinct().count()
display(distinct_rows)


1000

In [129]:
Dup_records=spark.sql("""
select `Organization Id`,count(*) from temp_emp
group by `Organization Id`
having count(*)>1
""")
Dup_records.show()

+---------------+--------+
|Organization Id|count(1)|
+---------------+--------+
+---------------+--------+



In [121]:
window_spec = Window.partitionBy("Country").orderBy(F.col("Number of employees").desc())
df_highest_salary = df_csv.withColumn("rank", F.dense_rank().over(window_spec)).filter(F.col("rank") == 1)
df_highest_salary.show()

+-----+---------------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+-------------------+----+
|Index|Organization Id|                Name|             Website|             Country|         Description|Founded|            Industry|Number of employees|rank|
+-----+---------------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+-------------------+----+
|   13|4EB9d3E5cF79b91|Bernard, Payne an...|http://williamson...|         Afghanistan|Polarized dynamic...|   1990|      Transportation|                730|   1|
|  578|3D41753AADAc2FB|       Mueller Group|https://www.nicho...|             Albania|Function-based va...|   1972|Graphic Design / ...|                956|   1|
|    2|AAC4f9aBF86EAeF|       Orr-Armstrong|https://www.chapm...|             Algeria|Ergonomic radical...|   1970|     Import / Export|               7994|   1|
|  467|2FaB6dAd4AfE024| Marq